# Assignment 8 – E-Commerce Analytics System


## Part 1: Data Generation

This section generates realistic e-commerce datasets using Python and Faker. The generated datasets include Customers, Products, Orders, and Order Items with intentional data quality issues for cleaning, validation, and analysis.

In [0]:
# Install Faker library 

%pip install faker

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
# ==========================================
# Import Required Libraries
# ==========================================

import os
import random
import pandas as pd

from faker import Faker
from datetime import datetime, timedelta

fake = Faker("en_IN")

Faker.seed(42)
random.seed(42)


In [0]:
# ==========================================
# Generate Customers Dataset
# ==========================================

# Empty list to store customer records
customers = []

# Customer Types
customer_types = ["REGULAR", "PREMIUM", "VIP"]

# Generate 500 Customers
for i in range(1, 501):

    customer_id = f"CUST{i:04d}"

    customer_name = fake.name()

    email = fake.email()

    # Registration Date (Last 3 Years)
    registration_date = fake.date_between(
        start_date="-3y",
        end_date="today"
    )

    # Random Customer Type
    customer_type = random.choice(customer_types)

    # Store Record
    customers.append([
        customer_id,
        customer_name,
        email,
        registration_date,
        customer_type
    ])

# Create DataFrame
customers_df = pd.DataFrame(
    customers,
    columns=[
        "customer_id",
        "customer_name",
        "email",
        "registration_date",
        "customer_type"
    ]
)

# ==========================================
# Introduce Invalid Emails (2%)
# ==========================================

# Select 10 random rows (2% of 500)
invalid_rows = random.sample(range(500), 10)

# Remove '@' from email to make it invalid
for i in invalid_rows:
    customers_df.loc[i, "email"] = (
        customers_df.loc[i, "email"].replace("@", "")
    )

# ==========================================
# Display Sample Data
# ==========================================

display(customers_df.head())

print("Total Customers :", len(customers_df))
print("Invalid Emails :", len(invalid_rows))

customer_id,customer_name,email,registration_date,customer_type
CUST0001,Aryan Maharaj,udantdewan@example.net,2023-11-02,REGULAR
CUST0002,Pahal Balay,chandertejas@example.org,2025-01-16,VIP
CUST0003,Rushil Saini,saumyamall@example.org,2025-10-21,REGULAR
CUST0004,Pahal Oak,nachiket35@example.org,2023-10-22,VIP
CUST0005,Jagrati Padmanabhan,caleb78@example.org,2026-06-13,VIP


Total Customers : 500
Invalid Emails : 10


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.ecommerce_analytics_system;

In [0]:
# ==========================================
# Save Customers Dataset
# ==========================================

customers_df.to_csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/customers.csv",
    index=False
)

print("customers.csv saved successfully.")

customers.csv saved successfully.


In [0]:
# ==========================================
# Generate Products Dataset
# ==========================================

# Categories and Subcategories
categories = {
    "Electronics": ["Mobile", "Laptop", "Headphones", "Camera"],
    "Clothing": ["Shirt", "Jeans", "Jacket", "Shoes"],
    "Home": ["Chair", "Table", "Sofa", "Lamp"],
    "Books": ["Novel", "Education", "Comics", "Biography"]
}

# Empty list to store product records
products = []

# Generate 500 Products
for i in range(1, 501):


    product_id = f"PROD{i:04d}"

    category = random.choice(list(categories.keys()))

    subcategory = random.choice(categories[category])

    product_name = fake.word().capitalize() + " " + subcategory

    cost_price = random.randint(100, 50000)

    # Store Record
    products.append([
        product_id,
        product_name,
        category,
        subcategory,
        cost_price
    ])

# Create DataFrame
products_df = pd.DataFrame(
    products,
    columns=[
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "cost_price"
    ]
)

# ==========================================
# Introduce Dirty Product Names
# ==========================================

rows = random.sample(range(500), 20)

for i in rows:

    # Add extra spaces
    if i % 2 == 0:
        products_df.loc[i, "product_name"] = (
            "   " + products_df.loc[i, "product_name"] + "   "
        )

    # Convert to Mixed Case
    else:
        products_df.loc[i, "product_name"] = (
            products_df.loc[i, "product_name"].swapcase()
        )

# ==========================================
# Display Sample Data
# ==========================================

display(products_df.head())

print("Total Products :", len(products_df))

product_id,product_name,category,subcategory,cost_price
PROD0001,Commodi Chair,Home,Chair,48785
PROD0002,iD jEANS,Clothing,Jeans,14280
PROD0003,Nulla Comics,Books,Comics,47586
PROD0004,Modi Chair,Home,Chair,12821
PROD0005,Nam Table,Home,Table,23750


Total Products : 500


In [0]:
# ==========================================
# Save Products Dataset
# ==========================================

products_df.to_csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/products.csv",
    index=False
)

print("products.csv saved successfully.")

products.csv saved successfully.


In [0]:
# ==========================================
# Generate Orders Dataset
# ==========================================

statuses = ["PLACED","SHIPPED","DELIVERED","CANCELLED","RETURNED"]

regions = ["NORTH","SOUTH","EAST","WEST"]

# Empty list to store orders
orders = []

for i in range(1, 701):

    order_id = f"ORD{i:05d}"

    customer_id = f"CUST{random.randint(1, 500):04d}"

    order_date = fake.date_time_between(
        start_date="-2y",
        end_date="now"
    )

    # Random order status
    status = random.choice(statuses)

    # Random region
    region = random.choice(regions)

    # Store order record
    orders.append([order_id,customer_id,order_date,status,region])

# Create DataFrame
orders_df = pd.DataFrame(
    orders,
    columns=[
        "order_id",
        "customer_id",
        "order_date",
        "status",
        "region_code"
    ]
)

# ==========================================
# Introduce NULL Customer IDs (5%)
# ==========================================

# 5% of 700 = 35 rows
null_rows = random.sample(range(700), 35)

orders_df.loc[
    null_rows,
    "customer_id"
] = None

# ==========================================
# Introduce Wrong Date Format
# ==========================================

# Select 20 random rows
wrong_dates = random.sample(range(700), 20)

# Convert selected dates into DD-MM-YYYY format
for i in wrong_dates:

    orders_df.loc[i, "order_date"] = (
        pd.to_datetime(
            orders_df.loc[i, "order_date"]
        ).strftime("%d-%m-%Y")
    )

# ==========================================
# Display Sample Data
# ==========================================

display(orders_df.head())

print("Total Orders:", len(orders_df))
print("NULL Customer IDs:", orders_df["customer_id"].isna().sum())
print("Wrong Date Format Rows:", len(wrong_dates))

order_id,customer_id,order_date,status,region_code
ORD00001,CUST0055,2025-01-19T13:13:20.248Z,SHIPPED,EAST
ORD00002,CUST0004,2025-05-06T05:17:58.825Z,CANCELLED,EAST
ORD00003,CUST0431,2024-09-28T14:06:12.044Z,RETURNED,WEST
ORD00004,CUST0412,2024-09-03T16:38:48.924Z,DELIVERED,WEST
ORD00005,CUST0172,2025-02-12T18:17:33.284Z,CANCELLED,SOUTH


Total Orders: 700
NULL Customer IDs: 35
Wrong Date Format Rows: 20


In [0]:
# ==========================================
# Save Orders Dataset
# ==========================================

orders_df.to_csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/orders.csv",
    index=False
)

print("orders.csv saved successfully.")

orders.csv saved successfully.


In [0]:
# ==========================================
# Generate Order Items Dataset
# ==========================================

order_items = []

for i in range(1, 2501):

    item_id = f"ITEM{i:05d}"

    order_id = f"ORD{random.randint(1,700):05d}"

    product_id = f"PROD{random.randint(1,500):04d}"

    quantity = random.randint(1,5)

    unit_price = random.randint(200,60000)

    discount_percent = random.randint(0,100)

    # Store Record
    order_items.append([
        item_id,
        order_id,
        product_id,
        quantity,
        unit_price,
        discount_percent
    ])

# Create DataFrame
order_items_df = pd.DataFrame(
    order_items,
    columns=[
        "item_id",
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount_percent"
    ]
)

# ==========================================
# Introduce Negative Quantity (3%)
# ==========================================

negative_rows = random.sample(range(2500), 75)

for i in negative_rows:

    order_items_df.loc[i, "quantity"] = -random.randint(1,5)

# ==========================================
# Display Sample Data
# ==========================================

display(order_items_df.head())

print("Total Order Items :", len(order_items_df))
print("Negative Quantity Rows :", len(negative_rows))

item_id,order_id,product_id,quantity,unit_price,discount_percent
ITEM00001,ORD00564,PROD0315,1,10734,49
ITEM00002,ORD00652,PROD0003,1,54152,18
ITEM00003,ORD00483,PROD0475,5,28732,11
ITEM00004,ORD00066,PROD0135,2,24503,10
ITEM00005,ORD00624,PROD0020,4,26456,78


Total Order Items : 2500
Negative Quantity Rows : 75


In [0]:
# ==========================================
# Save Order Items Dataset
# ==========================================

order_items_df.to_csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/order_items.csv",
    index=False
)

print("order_items.csv saved successfully.")

order_items.csv saved successfully.
